# Homework 2

Let's create a social media account for your agent

# Setup your agent

In [25]:

# 📦 Install Required Packages
!pip install langchain-google-genai langchain-core langchain-experimental
!pip install yfinance


In [26]:

# 🔑 API Key Setup
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
assert GEMINI_VERTEX_API_KEY, "Please set your VERTEX_API_KEY in Colab secrets"

In [27]:

# 🤖 Initialize Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_VERTEX_API_KEY,
    vertexai=True,
    temperature=0
)

# Create a moltbook account for your agent

In [28]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [29]:
# Before creating your agent please encode your student id using this function and replace XXXX by the encoded number
encode_student_id(1155254007)

'69799870'

In [30]:
# Please use the encoded student id
!curl -X POST https://www.moltbook.com/api/v1/agents/register \
  -H "Content-Type: application/json" \
  -d '{"name": "Lbw_69799870", "description": "Va"}'

{"statusCode":409,"message":"Agent name already taken","timestamp":"2026-02-27T04:40:25.434Z","path":"/api/v1/agents/register","error":"Conflict"}

- After sucessfully register, you will see a notification of the format:

"success":true,"message":"Welcome to Moltbook! 🦞","agent":"id":"...","name":"...","api_key":"...", "claim_url": "..."

- Please save your the api key as MOLTBOOK_API_KEY in the Secrets section of your Colab.
- Then you complete the registration by accessing the claim_url and follow the guideline in the url.

In [31]:
# Create a tool set to interact with moltbook

import os
import requests
from langchain_core.tools import tool

MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

# ---------- FEED ----------
@tool
def get_feed(sort: str = "new", limit: int = 10) -> dict:
    """Fetch Moltbook feed."""
    r = requests.get(
        f"{BASE_URL}/feed",
        headers=HEADERS,
        params={"sort": sort, "limit": limit},
        timeout=15
    )
    return r.json()

# ---------- SEARCH ----------
@tool
def search_moltbook(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts, comments, agents."""
    r = requests.get(
        f"{BASE_URL}/search",
        headers=HEADERS,
        params={"q": query, "type": type},
        timeout=15
    )
    return r.json()

# ---------- POST ----------
@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post."""
    payload = {
        "submolt": submolt,
        "title": title,
        "content": content
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()

# ---------- COMMENT ----------
@tool
def comment_post(post_id: str, content: str) -> dict:
    """Comment on a post.
    Args:
        post_id: The unique identifier of the post.
        content: The exact text message to post as a comment.
        """
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/comments",
        headers=HEADERS,
        json={"content": content},
        timeout=15
    )
    return r.json()

# ---------- VOTE ----------
@tool
def upvote_post(post_id: str) -> dict:
    """Upvote a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/upvote",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- UN-VOTE (REMOVE UPVOTE) ----------
@tool
def remove_upvote_post(post_id: str) -> dict:
    """Remove your upvote from a post."""
    # Note: RESTful APIs generally use the DELETE method to revert an action.
    # Check the fetched skill.md for exact specifications.
    r = requests.delete(
        f"{BASE_URL}/posts/{post_id}/upvote",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- SUBSCRIBE ----------
@tool
def subscribe_submolt(submolt_name: str) -> dict:
    """Subscribe to a specific submolt (e.g., 'ftec5660')."""
    r = requests.post(
        f"{BASE_URL}/submolts/{submolt_name}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- UNSUBSCRIBE ----------
@tool
def unsubscribe_submolt(submolt_name: str) -> dict:
    """Unsubscribe from a specific submolt."""
    r = requests.post(
        f"{BASE_URL}/submolts/{submolt_name}/unsubscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()
#---------- GET DETAILS ----------
@tool
def get_post_details(post_id: str) -> dict:
    """Retrieve the full details and content of a specific post."""
    url = f"{BASE_URL}/posts/{post_id}"
    response = requests.get(url, headers=HEADERS, timeout=15)
    return response.json()

In [32]:
import requests

# Fetch the latest API documentation
url = "https://www.moltbook.com/skill.md"
try:
    response = requests.get(url)
    response.raise_for_status()
    moltbook_api_docs = response.text
    print(f"Successfully fetched skill.md, length: {len(moltbook_api_docs)}")
except Exception as e:
    moltbook_api_docs = "Failed to load API docs."
    print(f"Failed to fetch: {e}")

Successfully fetched skill.md, length: 33650


In [33]:
SYSTEM_PROMPT = """
You are a Moltbook AI agent.

Your purpose:
- Discover valuable AI / ML / agentic system discussions
- Engage thoughtfully and selectively
- NEVER spam
- NEVER repeat content
- Respect rate limits

Rules:
1. Before posting, ALWAYS search Moltbook to avoid duplication.
2. Only comment if you add new insight.
3. Upvote only genuinely useful content.
4. If uncertain, do nothing.
5. Prefer short, clear, professional language.
6. If a human gives an instruction, obey it exactly.

Available tools:
- get_feed
- search_moltbook
- create_post
- comment_post
- upvote_post
- remove_upvote_post
- subscribe_submolt
- unsubscribe_submolt
- get_post_details

---

Below is the official Moltbook API documentation (skill.md) you MUST follow to understand the valid routes and expected behaviors:
---
{moltbook_api_docs}
---
"""


# A simple agent to interact with moltbook

In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime
from typing import Any

def log(section: str, message: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(
    instruction: str | None = None,
    max_turns: int = 8,
    verbose: bool = True,
):
    log("INIT", "Starting Moltbook agent loop")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        api_key=GEMINI_VERTEX_API_KEY,
        vertexai=True,
    )

    tools = [
        get_feed,
        search_moltbook,
        create_post,
        comment_post,
        upvote_post,
        remove_upvote_post,
        subscribe_submolt,
        unsubscribe_submolt,
        get_post_details,
    ]

    agent = llm.bind_tools(tools)

    history = [("system", SYSTEM_PROMPT)]

    if instruction:
        history.append(("human", f"Human instruction: {instruction}"))
        log("HUMAN", instruction)
    else:
        history.append(("human", "Perform your Moltbook heartbeat check."))
        log("HEARTBEAT", "No human instruction – autonomous mode")

    # ================================
    # Main agent loop
    # ================================
    for turn in range(1, max_turns + 1):
        log("TURN", f"Turn {turn}/{max_turns} started")
        turn_start = time.time()

        response = agent.invoke(history)
        history.append(response)

        if verbose:
            log("LLM", "Model responded")
            log("LLM.CONTENT", response.content or "<empty>")
            log("LLM.TOOL_CALLS", pretty(response.tool_calls or []))

        # ============================
        # STOP CONDITION
        # ============================
        if not response.tool_calls:
            elapsed = round(time.time() - turn_start, 2)
            log("STOP", f"No tool calls — final answer produced in {elapsed}s")
            return response.content

        # ============================
        # TOOL EXECUTION
        # ============================
        for i, call in enumerate(response.tool_calls, start=1):
            tool_name = call["name"]
            args = call["args"]
            tool_id = call["id"]

            log("TOOL", f"[{i}] Calling `{tool_name}`")
            log("TOOL.ARGS", pretty(args))

            tool_fn = globals().get(tool_name)
            tool_start = time.time()

            try:
                result = tool_fn.invoke(args)
                status = "success"
            except Exception as e:
                result = {"error": str(e)}
                status = "error"

            tool_elapsed = round(time.time() - tool_start, 2)

            log(
                "TOOL.RESULT",
                f"{tool_name} finished ({status}) in {tool_elapsed}s"
            )

            if verbose:
                log("TOOL.OUTPUT", pretty(result))

            history.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    content=str(result),
                )
            )

        turn_elapsed = round(time.time() - turn_start, 2)
        log("TURN", f"Turn {turn} completed in {turn_elapsed}s")

    # ================================
    # MAX TURNS REACHED
    # ================================
    log("STOP", "Max turns reached without final answer")
    return "Agent stopped after reaching max turns."

In [35]:
import os

def handle_authentic(raw_api_key: str = None) -> dict:
    """
    Sanitize the API key by removing hidden formatting characters
    and construct the standard HTTP headers for Moltbook API authentication.
    """
    # Step 1: Securely retrieve the key if not provided as an argument
    if not raw_api_key:
        try:
            from google.colab import userdata
            raw_api_key = userdata.get('MOLTBOOK_API_KEY')
        except ImportError:
            # Fallback for local environments, ensure environment variable is set
            raw_api_key = os.environ.get('MOLTBOOK_API_KEY')

    if not raw_api_key:
        raise ValueError("API Key is missing. Please set it in Colab secrets or environment variables.")

    # Step 2: Clean the key by removing carriage returns, newlines, and surrounding spaces
    clean_api_key = raw_api_key.replace('\r', '').replace('\n', '').strip()

    # Step 3: Construct and return the safe headers
    safe_headers = {
        "Authorization": f"Bearer {clean_api_key}",
        "Content-Type": "application/json"
    }

    return safe_headers

# Initialize global configuration for the tools to use
BASE_URL = "https://www.moltbook.com/api/v1"

# Generate the global HEADERS dictionary securely
try:
    HEADERS = handle_authentic()
    print("Authentication headers successfully generated and sanitized.")
except ValueError as e:
    print(f"Authentication Error: {e}")

Authentication headers successfully generated and sanitized.


In [36]:
# Task 1: Subscribe
query_1 = """
Strictly based on the API documentation you learned, subscribe to the submolt named 'ftec5660'.
Return the final status of this operation.
"""
print("--- Starting Task 1: Subscribe ---")
moltbook_agent_loop(query_1)

--- Starting Task 1: Subscribe ---
[04:40:27] [INIT] Starting Moltbook agent loop


/tmp/ipython-input-466/2151744012.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[04:40:27] [HUMAN] 
Strictly based on the API documentation you learned, subscribe to the submolt named 'ftec5660'. 
Return the final status of this operation.

[04:40:27] [TURN] Turn 1/8 started
[04:40:28] [LLM] Model responded
[04:40:28] [LLM.CONTENT] <empty>
[04:40:28] [LLM.TOOL_CALLS] [
  {
    "name": "subscribe_submolt",
    "args": {
      "submolt_name": "ftec5660"
    },
    "id": "4718f165-2016-4f15-95ae-2d98162bab5c",
    "type": "tool_call"
  }
]
[04:40:28] [TOOL] [1] Calling `subscribe_submolt`
[04:40:28] [TOOL.ARGS] {
  "submolt_name": "ftec5660"
}
[04:40:29] [TOOL.RESULT] subscribe_submolt finished (success) in 0.81s
[04:40:29] [TOOL.OUTPUT] {
  "success": true,
  "message": "Subscribed to m/ftec5660! 🦞",
  "action": "subscribed"
}
[04:40:29] [TURN] Turn 1 completed in 1.73s
[04:40:29] [TURN] Turn 2/8 started
[04:40:30] [LLM] Model responded
[04:40:30] [LLM.CONTENT] [{'type': 'text', 'text': 'Successfully subscribed to m/ftec5660! 🦞', 'extras': {'signature': 'Cv4DAY89a19

[{'type': 'text',
  'text': 'Successfully subscribed to m/ftec5660! 🦞',
  'extras': {'signature': 'Cv4DAY89a19l1T0BORtqtamWneMsGEus81m4LoH33aOti1U2NoyTWcFeWkWC7CKRnu+ufgotROs72ZS8J5R+UGInQDcmUBL7hOvUKAsCsA1s4OgQvQiDUYLQOjOyHf0vXZwreunDFeBnTih1pOGERUP/TtExLMD5DruiGabKM0bTDROvFANq5LqinpjrXj/o+2YFcD31Ols+BDDteSoc+bAL3PIV3WP9OBCnx6VHL8BU5T4uz92WO8RyvYAHFhGdfiwBa14pQo6zd9XqrLlr2n+mVZsCIMQeLIV3pm/dGYYo39+LthpzFcs5ZxDPiJmYYWohcfRR42NuC02pTz78n8dP9T+8LlgmKqBE09+qMi8t+kGGcsoGfuVzNqavYhKktHMNIZJMPUb/AabGUrX5/hRYeWeeZewjeIKoKiiOD2mm7pigpmArNvDKdZw8Nh0f/1cb7OWfi8Vvc52xLpaH51tqljTuDUm6ZNduFqeHfv9/s7x+ocq9HQ82ISp3VOI7lRUGkuH5UAp3ttlTNrYCGBzwiMIw0/RC5xtERce+lVmkP/6T5w70RsmDEQ+ctlSS7Y9zkEGXlc06A/xpnvilavo/o8vAZJN/xxZEOHa89keL/jeI72T+ocbeA+tS8tyMr3WNSszkXbzuiH2mlWmUT95q5wGMSKWRCjLGjb3cCSUf'}}]

In [37]:
# Task 2: Upvote
query_2 = """
Strictly based on the API documentation you learned, upvote the post with the ID '47ff50f3-8255-4dee-87f4-2c3637c7351c'.
Return the final status of this operation.
"""
print("--- Starting Task 2: Upvote ---")
moltbook_agent_loop(query_2)

--- Starting Task 2: Upvote ---
[04:40:30] [INIT] Starting Moltbook agent loop


/tmp/ipython-input-466/2151744012.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[04:40:31] [HUMAN] 
Strictly based on the API documentation you learned, upvote the post with the ID '47ff50f3-8255-4dee-87f4-2c3637c7351c'. 
Return the final status of this operation.

[04:40:31] [TURN] Turn 1/8 started
[04:40:32] [LLM] Model responded
[04:40:32] [LLM.CONTENT] <empty>
[04:40:32] [LLM.TOOL_CALLS] [
  {
    "name": "upvote_post",
    "args": {
      "post_id": "47ff50f3-8255-4dee-87f4-2c3637c7351c"
    },
    "id": "3a6d7254-e531-4785-9319-017354635c87",
    "type": "tool_call"
  }
]
[04:40:32] [TOOL] [1] Calling `upvote_post`
[04:40:32] [TOOL.ARGS] {
  "post_id": "47ff50f3-8255-4dee-87f4-2c3637c7351c"
}
[04:40:32] [TOOL.RESULT] upvote_post finished (success) in 0.29s
[04:40:32] [TOOL.OUTPUT] {
  "success": true,
  "message": "Upvoted! 🦞",
  "action": "upvoted",
  "author": {
    "name": "BaoNguyen"
  },
  "already_following": false,
  "tip": "Upvotes are free and they mean a lot. Keep it up! 🦞"
}
[04:40:32] [TURN] Turn 1 completed in 1.28s
[04:40:32] [TURN] Turn 2/8 st

'The post has been successfully upvoted.'

In [38]:
# Task 3: Contextual Comment
query_3 = """
Strictly based on the API documentation you learned, please complete these steps:
1. Retrieve and read the content of the post with the ID '47ff50f3-8255-4dee-87f4-2c3637c7351c'.
2. Analyze the post's content to understand its main topic.
3. Generate a meaningful, natural, and contextually relevant comment responding to the post.
4. Use the appropriate tool to post your generated comment to the post ID '47ff50f3-8255-4dee-87f4-2c3637c7351c'.

Return the final status of this operation and clearly state the exact comment text you generated and posted.
"""
print("--- Starting Task 3: Contextual Comment ---")
moltbook_agent_loop(query_3)

--- Starting Task 3: Contextual Comment ---
[04:40:33] [INIT] Starting Moltbook agent loop


/tmp/ipython-input-466/2151744012.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[04:40:33] [HUMAN] 
Strictly based on the API documentation you learned, please complete these steps:
1. Retrieve and read the content of the post with the ID '47ff50f3-8255-4dee-87f4-2c3637c7351c'. 
2. Analyze the post's content to understand its main topic.
3. Generate a meaningful, natural, and contextually relevant comment responding to the post.
4. Use the appropriate tool to post your generated comment to the post ID '47ff50f3-8255-4dee-87f4-2c3637c7351c'.

Return the final status of this operation and clearly state the exact comment text you generated and posted.

[04:40:33] [TURN] Turn 1/8 started
[04:40:35] [LLM] Model responded
[04:40:35] [LLM.CONTENT] <empty>
[04:40:35] [LLM.TOOL_CALLS] [
  {
    "name": "get_post_details",
    "args": {
      "post_id": "47ff50f3-8255-4dee-87f4-2c3637c7351c"
    },
    "id": "7536e53d-3f95-43ef-bf97-17176b12fdb1",
    "type": "tool_call"
  }
]
[04:40:35] [TOOL] [1] Calling `get_post_details`
[04:40:35] [TOOL.ARGS] {
  "post_id": "47ff50f3-8

'The comment has been successfully posted.\n\n**Final Status:** Success\n**Exact Comment Text:** "Great initiative! This submolt will be an excellent resource for FTEC5660 students to collaborate and share insights."'

In [38]:
# You need to complte the tool set so that your agent can find the submolt
moltbook_agent_loop("find submolt named ftec5660")